# U21 — ETL & Orchestration: Lab

### Real-world brief: an ETL pipeline for a wind farm's SCADA & maintenance data

A wind-farm operator has three **source systems**, all imperfect: hourly **SCADA telemetry** (with gaps, nulls, duplicates and bad sensor values), an **asset registry**, and a **maintenance log** (in Excel). The analytics team needs a clean **daily performance table** in a warehouse — and it must be safe to re-run, load only new data, and fail loudly when the data is wrong.

You'll build the full pipeline: **Extract → Transform → Quality-check → Load**, make it **idempotent** and **incremental**, and wrap it in a tiny **orchestration DAG** with retries — the real day-to-day of data engineering.

**Resources provided:** `turbine_telemetry.csv`, `turbine_registry.csv`, `maintenance_log.xlsx`. We load into a local **SQLite** file as the 'warehouse' (no server needed).

_Phase F — Data Engineering._

#objectives

Extract from heterogeneous sources (CSV + Excel)

Transform: clean, validate, join and aggregate to a fact table

Load to a warehouse with idempotent UPSERTs

Run incremental loads using a high-water mark

Add data-quality checks and orchestrate tasks as a DAG with retries

#how to use this lab

Worked demos teach the pattern; 🧪 LAB EXERCISE cells are real tasks — replace `# YOUR CODE HERE`. Run top to bottom with Shift+Enter.

In [1]:
# === SETUP: build the source files if missing ===
import os
import numpy as np
import pandas as pd


def build_windfarm(tele_path="turbine_telemetry.csv", reg_path="turbine_registry.csv",
                   maint_path="maintenance_log.xlsx", seed=211, verbose=False):
    """Raw operational data for a wind farm — the messy SOURCE systems an ETL pipeline must
    ingest (U21). Three sources, deliberately imperfect like real SCADA exports:

      turbine_telemetry.csv  hourly SCADA readings (with gaps, nulls, bad values, dupes)
      turbine_registry.csv   asset master (one row per turbine)
      maintenance_log.xlsx   work orders / downtime events
    """
    rng = np.random.default_rng(seed)

    # ---- asset registry (8 turbines across 2 sites) ----
    turbines = [f"T{i:02d}" for i in range(1, 9)]
    sites = ["Kutch", "Kutch", "Kutch", "Kutch", "Satara", "Satara", "Satara", "Satara"]
    models = rng.choice(["GE-2.0", "Vestas-2.5", "Suzlon-2.1"], 8)
    rated = np.array([2000, 2000, 2500, 2500, 2100, 2100, 2000, 2500])
    reg = pd.DataFrame({
        "turbine_id": turbines, "site": sites, "model": models,
        "rated_power_kw": rated,
        "commission_date": pd.to_datetime("2019-01-01") + pd.to_timedelta(rng.integers(0, 1500, 8), unit="D"),
        "latitude": np.round(rng.uniform(17, 23, 8), 4),
        "longitude": np.round(rng.uniform(69, 74, 8), 4),
    })
    reg.to_csv(reg_path, index=False)

    # ---- hourly telemetry over 30 days ----
    dates = pd.date_range("2024-05-01", "2024-05-30 23:00", freq="h")
    rows = []
    for ti, tid in enumerate(turbines):
        rp = rated[ti]
        wind = np.clip(rng.weibull(2.0, len(dates)) * 6.5, 0, 25)
        # power curve: cut-in 3, rated ~12 m/s, cut-out 25
        pc = np.clip((wind - 3) / (12 - 3), 0, 1) ** 3
        pc[wind < 3] = 0; pc[wind > 25] = 0
        power = pc * rp * rng.normal(1.0, 0.05, len(dates))
        power = np.clip(power, 0, rp * 1.05)
        rpm = np.clip(wind * 1.3 + rng.normal(0, 0.5, len(dates)), 0, 18)
        amb = 28 + 6 * np.sin(np.arange(len(dates)) / 24 * 2 * np.pi) + rng.normal(0, 1.5, len(dates))
        gearbox = amb + 0.018 * power + rng.normal(0, 2, len(dates))
        nacelle = amb + 0.010 * power + rng.normal(0, 1.5, len(dates))
        df = pd.DataFrame({
            "timestamp": dates, "turbine_id": tid,
            "wind_speed_ms": wind.round(2), "power_kw": power.round(1),
            "rotor_rpm": rpm.round(2), "gearbox_temp_c": gearbox.round(1),
            "nacelle_temp_c": nacelle.round(1),
        })
        rows.append(df)
    tele = pd.concat(rows, ignore_index=True)

    # ---- inject realistic messiness ----
    n = len(tele)
    # 1) missing rows (sensor dropout) -> availability gaps
    drop = rng.random(n) < 0.03
    tele = tele[~drop].reset_index(drop=True); n = len(tele)
    # 2) null sensor values
    for col in ["gearbox_temp_c", "rotor_rpm", "wind_speed_ms"]:
        tele.loc[rng.random(n) < 0.01, col] = np.nan
    # 3) impossible values
    bad = rng.random(n) < 0.004
    tele.loc[bad, "power_kw"] = rng.choice([-50, -999, 99999], bad.sum())
    # 4) duplicate rows
    dups = tele.sample(60, random_state=seed)
    tele = pd.concat([tele, dups], ignore_index=True)
    # 5) shuffle (timestamps arrive unsorted)
    tele = tele.sample(frac=1, random_state=seed).reset_index(drop=True)
    tele.to_csv(tele_path, index=False)

    # ---- maintenance work orders ----
    nwo = 120
    wo = pd.DataFrame({
        "work_order_id": [f"WO-{1000+i}" for i in range(nwo)],
        "turbine_id": rng.choice(turbines, nwo),
        "start_date": pd.to_datetime("2024-05-01") + pd.to_timedelta(rng.integers(0, 30, nwo), unit="D"),
        "type": rng.choice(["scheduled", "corrective"], nwo, p=[0.45, 0.55]),
        "downtime_hours": np.round(rng.gamma(2.0, 3.0, nwo), 1),
        "cost_inr": (rng.gamma(2.0, 25000, nwo)).round(0).astype(int),
    })
    wo["end_date"] = wo["start_date"] + pd.to_timedelta(wo["downtime_hours"], unit="h")
    wo = wo[["work_order_id", "turbine_id", "start_date", "end_date", "type", "downtime_hours", "cost_inr"]]
    wo.to_excel(maint_path, index=False, sheet_name="work_orders")

    if verbose:
        print("telemetry:", tele.shape, "| nulls:", int(tele.isna().sum().sum()),
              "| dupes:", int(tele.duplicated().sum()),
              "| bad power rows:", int(((tele.power_kw < 0) | (tele.power_kw > 5000)).sum()))
        print("registry:", reg.shape, "| maintenance:", wo.shape)
        print("date span:", tele.timestamp.min(), "->", tele.timestamp.max())
    return tele, reg, wo

if not (os.path.exists('turbine_telemetry.csv') and os.path.exists('turbine_registry.csv')
        and os.path.exists('maintenance_log.xlsx')):
    build_windfarm(); print('Generated source files.')
else:
    print('Found the provided source files.')

Generated source files.


In [2]:
import pandas as pd, numpy as np, sqlite3
pd.set_option('display.width', 120)
WAREHOUSE = 'windfarm_warehouse.db'   # our local 'data warehouse'

#1. Extract — read the raw sources

In [3]:
# -----------------------------------------------------------
# 🔹 1A. EXTRACT: pull each source and inspect it
# -----------------------------------------------------------
def extract():
    tele = pd.read_csv('turbine_telemetry.csv', parse_dates=['timestamp'])
    reg = pd.read_csv('turbine_registry.csv', parse_dates=['commission_date'])
    maint = pd.read_excel('maintenance_log.xlsx', parse_dates=['start_date', 'end_date'])
    return tele, reg, maint

tele, reg, maint = extract()
print('telemetry:', tele.shape, '| registry:', reg.shape, '| maintenance:', maint.shape)
print('\ntelemetry dtypes:'); print(tele.dtypes)
tele.head(3)

telemetry: (5649, 7) | registry: (8, 7) | maintenance: (120, 7)

telemetry dtypes:
timestamp         datetime64[ns]
turbine_id                object
wind_speed_ms            float64
power_kw                 float64
rotor_rpm                float64
gearbox_temp_c           float64
nacelle_temp_c           float64
dtype: object


,timestamp,turbine_id,wind_speed_ms,power_kw,rotor_rpm,gearbox_temp_c,nacelle_temp_c
0,2024-05-23 18:00:00,T07,6.23,91.7,7.65,23.4,19.1
1,2024-05-20 00:00:00,T08,5.32,47.5,7.26,26.6,26.6
2,2024-05-22 15:00:00,T08,7.27,259.2,9.30,26.7,26.7


In [4]:
# A quick data-health scan reveals the mess we must clean
print('null values per column:'); print(tele.isna().sum())
print('\nduplicate rows:', int(tele.duplicated().sum()))
print('power_kw range:', tele.power_kw.min(), 'to', tele.power_kw.max(), '(negatives & spikes = bad)')

null values per column:
timestamp          0
turbine_id         0
wind_speed_ms     44
power_kw           0
rotor_rpm         55
gearbox_temp_c    57
nacelle_temp_c     0
dtype: int64

duplicate rows: 60
power_kw range: -999.0 to 99999.0 (negatives & spikes = bad)


#2. Transform — clean, join, aggregate

In [5]:
# -----------------------------------------------------------
# 🔹 2A. CLEAN the telemetry
# -----------------------------------------------------------
def clean_telemetry(tele, reg):
    df = tele.drop_duplicates().copy()                       # 1) drop exact dupes
    df = df.merge(reg[['turbine_id', 'rated_power_kw', 'site']], on='turbine_id', how='left')
    # 2) remove impossible power values (negative or above rated headroom)
    df = df[(df.power_kw >= 0) & (df.power_kw <= df.rated_power_kw * 1.1)]
    # 3) drop rows missing the fields we need downstream
    df = df.dropna(subset=['power_kw', 'wind_speed_ms'])
    # 4) derive fields
    df['date'] = df['timestamp'].dt.date
    df['capacity_factor'] = df['power_kw'] / df['rated_power_kw']
    return df

clean = clean_telemetry(tele, reg)
print('rows after cleaning:', len(clean), '(from', len(tele), ')')
print('power_kw range now:', round(clean.power_kw.min(), 1), 'to', round(clean.power_kw.max(), 1))

rows after cleaning: 5527 (from 5649 )
power_kw range now: 0.0 to 2625.0


In [6]:
# -----------------------------------------------------------
# 🔹 2B. AGGREGATE to a daily fact table per turbine
# -----------------------------------------------------------
EXPECTED_PER_DAY = 24   # hourly readings -> 24 expected intervals/day
def build_daily_fact(clean, maint):
    g = clean.groupby(['turbine_id', 'date'])
    fact = g.agg(avg_power_kw=('power_kw', 'mean'),
                 energy_kwh=('power_kw', 'sum'),          # hourly kW summed = kWh
                 avg_wind_ms=('wind_speed_ms', 'mean'),
                 max_gearbox_c=('gearbox_temp_c', 'max'),
                 capacity_factor=('capacity_factor', 'mean'),
                 intervals=('power_kw', 'count')).reset_index()
    fact['availability_pct'] = (fact['intervals'] / EXPECTED_PER_DAY * 100).clip(upper=100).round(1)
    # join maintenance downtime per turbine per day
    m = maint.copy(); m['date'] = m['start_date'].dt.date
    dt = m.groupby(['turbine_id', 'date'])['downtime_hours'].sum().reset_index()
    fact = fact.merge(dt, on=['turbine_id', 'date'], how='left')
    fact['downtime_hours'] = fact['downtime_hours'].fillna(0)
    for c in ['avg_power_kw', 'energy_kwh', 'avg_wind_ms', 'capacity_factor']:
        fact[c] = fact[c].round(3)
    return fact

fact = build_daily_fact(clean, maint)
print('daily fact rows:', fact.shape)
fact.head()

daily fact rows: (240, 10)


,turbine_id,date,avg_power_kw,energy_kwh,avg_wind_ms,max_gearbox_c,capacity_factor,intervals,availability_pct,downtime_hours
0,T01,2024-05-01,514.774,11839.8,6.890,68.1,0.257,23,95.8,0.0
1,T01,2024-05-02,254.687,5857.8,5.925,60.6,0.127,23,95.8,0.0
2,T01,2024-05-03,346.770,7975.7,5.991,69.9,0.173,23,95.8,0.0
3,T01,2024-05-04,125.633,3015.2,4.892,44.0,0.063,24,100.0,0.0
4,T01,2024-05-05,156.696,3604.0,5.539,45.5,0.078,23,95.8,4.8


In [7]:
# 1. Add load_factor_flag and count underperforming days
fact['load_factor_flag'] = np.where(fact['capacity_factor'] < 0.25, 'underperforming', 'normal')
underperforming_days = (fact['load_factor_flag'] == 'underperforming').sum()
print(f'Number of underperforming turbine-days: {underperforming_days}')

# 2. Why derive in transform:
# Deriving flags like 'load_factor_flag' in the transform step ensures that all downstream applications
# (e.g., dashboards, reports, further analyses) use a consistent definition of 'underperforming'.
# It simplifies queries and calculations for end-users, as they don't need to re-implement the logic
# each time. This approach also centralizes business logic, making the pipeline more robust,
# easier to maintain, and less prone to inconsistencies across different data consumers.

Number of underperforming turbine-days: 237


#3. Load — write to the warehouse, idempotently

In [8]:
# -----------------------------------------------------------
# 🔹 3A. CREATE the warehouse table with a primary key
# The PK (turbine_id, date) is what makes UPSERT possible.
# -----------------------------------------------------------
def init_warehouse(db=WAREHOUSE):
    con = sqlite3.connect(db)
    con.execute('''CREATE TABLE IF NOT EXISTS daily_performance (
        turbine_id TEXT, date TEXT, avg_power_kw REAL, energy_kwh REAL,
        avg_wind_ms REAL, max_gearbox_c REAL, capacity_factor REAL,
        intervals INTEGER, availability_pct REAL, downtime_hours REAL,
        PRIMARY KEY (turbine_id, date) )''')
    con.commit(); con.close()

init_warehouse()
print('warehouse table ready.')

warehouse table ready.


In [9]:
# -----------------------------------------------------------
# 🔹 3B. IDEMPOTENT LOAD via UPSERT (INSERT ... ON CONFLICT DO UPDATE)
# Re-running with the same data must NOT create duplicates.
# -----------------------------------------------------------
def upsert_fact(fact, db=WAREHOUSE):
    cols = ['turbine_id','date','avg_power_kw','energy_kwh','avg_wind_ms','max_gearbox_c',
            'capacity_factor','intervals','availability_pct','downtime_hours']
    rows = fact.assign(date=fact['date'].astype(str))[cols].itertuples(index=False, name=None)
    sql = f'''INSERT INTO daily_performance ({','.join(cols)})
              VALUES ({','.join(['?']*len(cols))})
              ON CONFLICT(turbine_id, date) DO UPDATE SET
              {', '.join(f'{c}=excluded.{c}' for c in cols[2:])}'''
    con = sqlite3.connect(db)
    con.executemany(sql, rows); con.commit()
    n = con.execute('SELECT COUNT(*) FROM daily_performance').fetchone()[0]
    con.close(); return n

n1 = upsert_fact(fact)
n2 = upsert_fact(fact)   # run AGAIN — idempotent, so the count must stay the same
print('rows after first load:', n1)
print('rows after second load:', n2, '(unchanged -> idempotent!)')

rows after first load: 240
rows after second load: 240 (unchanged -> idempotent!)


In [12]:
# 1-2. query, modify, upsert, re-query; confirm update-not-insert

# Pick a turbine-day to modify (e.g., first row of fact)
turbine_to_modify = fact.iloc[0]
turbine_id_val = turbine_to_modify['turbine_id']
date_val = str(turbine_to_modify['date']) # Convert date to string for SQL query

# Query its energy_kwh from the warehouse before modification
con = sqlite3.connect(WAREHOUSE)
initial_energy = con.execute(
    "SELECT energy_kwh FROM daily_performance WHERE turbine_id = ? AND date = ?",
    (turbine_id_val, date_val)
).fetchone()[0]
print(f"Initial energy_kwh for {turbine_id_val} on {date_val}: {initial_energy}")

# Create a copy of fact and change the energy_kwh for the chosen turbine-day
fact_copy = fact.copy()
new_energy_kwh = 9999.999 # A distinct new value
fact_copy.loc[(fact_copy['turbine_id'] == turbine_id_val) & (fact_copy['date'].apply(str) == date_val), 'energy_kwh'] = new_energy_kwh

# Upsert the modified fact_copy
rows_before_upsert = con.execute("SELECT COUNT(*) FROM daily_performance").fetchone()[0]
upsert_fact(fact_copy)
rows_after_upsert = con.execute("SELECT COUNT(*) FROM daily_performance").fetchone()[0]

# Re-query the energy_kwh and total row count to confirm update and no duplication
updated_energy = con.execute(
    "SELECT energy_kwh FROM daily_performance WHERE turbine_id = ? AND date = ?",
    (turbine_id_val, date_val)
).fetchone()[0]
con.close()

print(f"Updated energy_kwh for {turbine_id_val} on {date_val}: {updated_energy}")
print(f"Rows before upsert: {rows_before_upsert}, Rows after upsert: {rows_after_upsert}")

# Assertions to confirm behavior
assert updated_energy == new_energy_kwh
assert rows_before_upsert == rows_after_upsert
print("Confirmation: Row was updated, not duplicated, and total row count is unchanged.")

# 3. why idempotency enables safe retries:
# Idempotency ensures that executing an operation multiple times has the same effect as executing it once.
# In a data pipeline, if an 'upsert' step is idempotent, a pipeline failure and subsequent retry will not lead
# to duplicate data or incorrect states. This allows for safe retries of failed pipeline runs without manual
# intervention to clean up partially processed data, making the pipeline more robust and resilient to transient
# errors. If the upsert were not idempotent, retries could lead to duplicate records, causing data integrity issues.

Initial energy_kwh for T01 on 2024-05-01: 11839.8
Updated energy_kwh for T01 on 2024-05-01: 9999.999
Rows before upsert: 240, Rows after upsert: 240
Confirmation: Row was updated, not duplicated, and total row count is unchanged.


#4. Incremental loads with a high-water mark

In [13]:
# -----------------------------------------------------------
# 🔹 4A. Only process dates NEWER than what's already loaded
# -----------------------------------------------------------
def high_water_mark(db=WAREHOUSE):
    con = sqlite3.connect(db)
    hw = con.execute('SELECT MAX(date) FROM daily_performance').fetchone()[0]
    con.close(); return hw

def run_incremental(db=WAREHOUSE):
    tele, reg, maint = extract()
    clean = clean_telemetry(tele, reg)
    fact = build_daily_fact(clean, maint)
    hw = high_water_mark(db)
    if hw is not None:
        before = len(fact)
        fact = fact[fact['date'].astype(str) > hw]   # keep only new partitions
        print(f'high-water mark = {hw}; {before} -> {len(fact)} new rows to load')
    return upsert_fact(fact, db) if len(fact) else None

# Simulate: warehouse already has everything, so an incremental run finds nothing new
print('high-water mark currently:', high_water_mark())
run_incremental()
print('A fresh run loads only new dates — cheap and fast.')

high-water mark currently: 2024-05-30
high-water mark = 2024-05-30; 240 -> 0 new rows to load
A fresh run loads only new dates — cheap and fast.


In [16]:
# 1-2. craft a new-date partition and load only the new rows

# Get current row count before backfill
con = sqlite3.connect(WAREHOUSE)
rows_before_backfill = con.execute('SELECT COUNT(*) FROM daily_performance').fetchone()[0]
con.close()
print(f"Rows in warehouse before backfill: {rows_before_backfill}")

# Create a copy of fact and shift all dates to a new, future date
fact_new_day = fact.copy()
# Dynamically determine a new date one day after the current high-water mark
current_hw = high_water_mark(WAREHOUSE)
if current_hw:
    new_date = pd.to_datetime(current_hw) + pd.Timedelta(days=1)
else:
    new_date = pd.to_datetime('2024-05-01') # Default start if no high-water mark
new_date = new_date.date() # Ensure it's a date object
fact_new_day['date'] = new_date

# Define a function to run an incremental load with a provided fact DataFrame
def run_incremental_with_fact(new_fact_df, db=WAREHOUSE):
    hw = high_water_mark(db)
    if hw is not None:
        # Ensure 'date' column is date objects for proper comparison
        new_fact_df['date'] = pd.to_datetime(new_fact_df['date'])
        filtered_fact = new_fact_df[new_fact_df['date'].astype(str) > hw]
        print(f'high-water mark = {hw}; {len(new_fact_df)} new rows in source -> {len(filtered_fact)} to load after filter')
        return upsert_fact(filtered_fact, db) if len(filtered_fact) else None
    else:
        # If no high-water mark, load everything
        return upsert_fact(new_fact_df, db)

# Run the incremental load with the new-day data
rows_loaded_by_backfill = run_incremental_with_fact(fact_new_day)

# Get row count after backfill
con = sqlite3.connect(WAREHOUSE)
rows_after_backfill = con.execute('SELECT COUNT(*) FROM daily_performance').fetchone()[0]
con.close()
print(f"Rows in warehouse after backfill: {rows_after_backfill}")

# Confirm the row count grew by exactly the number of new turbine-days
# Since fact_new_day assigns one new date to ALL original rows, and the PK is (turbine_id, date),
# only one row per unique turbine_id for the new date will be inserted.
expected_increase = fact_new_day['turbine_id'].nunique()
assert rows_after_backfill == rows_before_backfill + expected_increase
print(f"Confirmation: Warehouse grew by {expected_increase} rows, as expected.")

# 3. why incremental loads matter at scale:
# Incremental loads are crucial for tables with billions of rows because full reloads become prohibitively
# expensive and slow. Loading only new or changed data (the 'increment') drastically reduces the amount of
# data processed and transferred, saving computational resources (CPU, memory, disk I/O) and time.
# This efficiency enables pipelines to run more frequently, providing fresher data to downstream applications
# and ensuring that ETL processes can keep up with ever-growing data volumes without impacting performance
# or exceeding resource budgets.

Rows in warehouse before backfill: 248
high-water mark = 2024-06-01; 240 new rows in source -> 240 to load after filter
Rows in warehouse after backfill: 256
Confirmation: Warehouse grew by 8 rows, as expected.


#5. Data-quality checks

In [19]:
# -----------------------------------------------------------
# 🔹 5A. A tiny quality-check framework — fail loudly on bad data
# -----------------------------------------------------------
class DataQualityError(Exception):
    pass

def run_quality_checks(fact, tele=None):
    checks = {
        'non_empty': len(fact) > 0,
        'keys_not_null': fact[['turbine_id', 'date']].notna().all().all(),
        'capacity_factor_in_range': fact['capacity_factor'].between(0, 1.1).all(),
        'availability_in_range': fact['availability_pct'].between(0, 100).all(),
        'no_negative_energy': (fact['energy_kwh'] >= 0).all(),
    }

    if tele is not None and not fact.empty:
        max_date_fact = pd.to_datetime(fact['date']).max().date()
        max_timestamp_tele = tele['timestamp'].max()
        max_date_tele = max_timestamp_tele.date()
        freshness_diff = max_date_tele - max_date_fact
        checks['data_is_fresh'] = freshness_diff <= pd.Timedelta(days=2)
    elif tele is not None and fact.empty:
        checks['data_is_fresh'] = False # If fact is empty, it's not fresh

    failed = [name for name, ok in checks.items() if not ok]
    for name, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    if failed:
        raise DataQualityError(f'quality checks failed: {failed}')
    return True

In [20]:
# Demonstrate data_is_fresh check failing
print("\n--- Demonstrating data_is_fresh check failure ---")
# Create a copy of fact with old dates
fact_stale = fact.copy()
# Shift all dates back by a significant amount to make it stale
fact_stale['date'] = pd.to_datetime(fact_stale['date']) - pd.Timedelta(days=10)

try:
    run_quality_checks(fact_stale, tele=tele)
except DataQualityError as e:
    print(f"Expected failure for stale data: {e}")

print("\n--- Running quality checks with fresh data (should pass) ---")
run_quality_checks(fact, tele=tele)
print('All quality checks passed for fresh data.')


--- Demonstrating data_is_fresh check failure ---
  [PASS] non_empty
  [PASS] keys_not_null
  [PASS] capacity_factor_in_range
  [PASS] availability_in_range
  [PASS] no_negative_energy
  [FAIL] data_is_fresh
Expected failure for stale data: quality checks failed: ['data_is_fresh']

--- Running quality checks with fresh data (should pass) ---
  [PASS] non_empty
  [PASS] keys_not_null
  [PASS] capacity_factor_in_range
  [PASS] availability_in_range
  [PASS] no_negative_energy
  [PASS] data_is_fresh
All quality checks passed for fresh data.


#6. Orchestration — a mini DAG with retries

In [24]:
# -----------------------------------------------------------
# 🔹 6A. Define tasks + dependencies and run them in order
# This is the essence of Airflow: a DAG of tasks, each retried on failure.
# -----------------------------------------------------------
import time, traceback
import random

# Global counter for the flaky task
flaky_task_failure_counter = 0

DAG = {
    'extract':       [],
    'transform':     ['extract'],
    'flaky_task':    ['transform'],
    'quality_check': ['flaky_task'],
    'load':          ['quality_check'],
    'notify':        ['load']
}

def topological_order(dag):
    order, seen = [], set()
    def visit(t):
        if t in seen: return
        for dep in dag[t]: visit(dep)
        seen.add(t); order.append(t)
    for t in dag: visit(t)
    return order

def run_dag(dag, tasks, max_retries=2):
    ctx = {}
    for t in topological_order(dag):
        for attempt in range(1, max_retries + 2):
            try:
                tasks[t](ctx)
                print(f'  [OK]   {t}')
                break
            except Exception as e:
                print(f'  [RETRY {attempt}] {t} failed: {e}')
                if attempt == max_retries + 1:
                    print(f'  [FAIL] {t} gave up after {attempt} attempts'); raise
                time.sleep(0.1)
    return ctx

print('task order:', topological_order(DAG))

task order: ['extract', 'transform', 'flaky_task', 'quality_check', 'load', 'notify']


In [25]:
# Wire the pipeline functions into DAG tasks (they share a context dict)
def task_extract(ctx):  ctx['tele'], ctx['reg'], ctx['maint'] = extract()
def task_transform(ctx): ctx['fact'] = build_daily_fact(clean_telemetry(ctx['tele'], ctx['reg']), ctx['maint'])

def task_flaky(ctx):
    global flaky_task_failure_counter
    if flaky_task_failure_counter < 1:
        flaky_task_failure_counter += 1
        raise ValueError("Simulating a transient failure!")
    print("Flaky task succeeded!")

def task_quality(ctx):   run_quality_checks(ctx['fact'], ctx['tele'])
def task_load(ctx):      ctx['rows'] = upsert_fact(ctx['fact'])
def task_notify(ctx):
    print(f"\nNotification: Pipeline completed successfully! {ctx['rows']} rows loaded into the warehouse.")

TASKS = {
    'extract': task_extract,
    'transform': task_transform,
    'flaky_task': task_flaky,
    'quality_check': task_quality,
    'load': task_load,
    'notify': task_notify
}

print('Running the pipeline DAG:')
ctx = run_dag(DAG, TASKS)
print('pipeline complete — warehouse rows:', ctx['rows'])

Running the pipeline DAG:
  [OK]   extract
  [OK]   transform
  [RETRY 1] flaky_task failed: Simulating a transient failure!
Flaky task succeeded!
  [OK]   flaky_task
  [PASS] non_empty
  [PASS] keys_not_null
  [PASS] capacity_factor_in_range
  [PASS] availability_in_range
  [PASS] no_negative_energy
  [PASS] data_is_fresh
  [OK]   quality_check
  [OK]   load

Notification: Pipeline completed successfully! 256 rows loaded into the warehouse.
  [OK]   notify
pipeline complete — warehouse rows: 256


#📘 Summary

| Stage | What you built |
| ----- | -------------- |
| Extract | read CSV + Excel sources, scanned data health |
| Transform | cleaned bad/dup/null rows, joined, aggregated to a daily fact |
| Load | idempotent UPSERT into a SQLite warehouse (re-runnable) |
| Incremental | high-water mark loads only new partitions |
| Quality | a check framework that fails loudly on bad data |
| Orchestrate | a DAG of tasks run in order, with retries |

**Core lesson:** a good pipeline is **reliability engineering** — idempotent so you can retry, incremental so it scales, validated so bad data is caught, and orchestrated so it runs itself. Tools like Airflow are this same DAG-of-tasks idea, productionised.

**Next — U22:** when the data outgrows one machine, move to streaming (Kafka) and distributed compute (Spark).